# **Atelier Préparation de Données Images**

## **Partie 1 – Exploration du dataset**

### **1.1 Développer un programme Python capable de récupérer, pour chaque image, son nom, sa classe, son format, son mode, sa largeur, sa hauteur, l’écart-type de ses pixels, son nombre de canaux et sa taille.**

In [77]:
import os
import numpy as np
import pandas as pd
from PIL import Image

RAW_DIR = "../data/raw"

In [78]:
def get_image_info(filepath, classe):
    """Récupère les infos d'une image. Gère aussi le cas d'un fichier corrompu."""
    nom = os.path.basename(filepath)
    taille_octets = os.path.getsize(filepath)

    try:
        with Image.open(filepath) as img:
            img.load()  # force la lecture complète des pixels (détecte les fichiers corrompus)
            pixels = np.array(img)
            return {
                "nom": nom,
                "classe": classe,
                "format": img.format,
                "mode": img.mode,
                "largeur": img.width,
                "hauteur": img.height,
                "ecart_type_pixels": pixels.std(),
                "nb_canaux": len(img.getbands()),
                "taille_octets": taille_octets,
                "corrompue": False,
            }
    except Exception:
        return {
            "nom": nom,
            "classe": classe,
            "format": None,
            "mode": None,
            "largeur": None,
            "hauteur": None,
            "ecart_type_pixels": None,
            "nb_canaux": None,
            "taille_octets": taille_octets,
            "corrompue": True,
        }

In [79]:
lignes = []

for classe in sorted(os.listdir(RAW_DIR)):
    dossier_classe = os.path.join(RAW_DIR, classe)
    for nom_fichier in sorted(os.listdir(dossier_classe)):
        chemin_image = os.path.join(dossier_classe, nom_fichier)
        lignes.append(get_image_info(chemin_image, classe))

df_images = pd.DataFrame(lignes)

print(f"Nombre total d'images : {len(df_images)}")
df_images.head()

Nombre total d'images : 1032


,nom,classe,format,mode,largeur,hauteur,ecart_type_pixels,nb_canaux,taille_octets,corrompue
0,cardboard1.jpg,cardboard,JPEG,RGB,512.0,384.0,40.588529,3.0,17333,False
1,cardboard10.jpg,cardboard,JPEG,RGB,512.0,384.0,42.571288,3.0,21683,False
2,cardboard100.jpg,cardboard,JPEG,RGB,512.0,384.0,46.108305,3.0,14884,False
3,cardboard101.jpg,cardboard,JPEG,RGB,512.0,384.0,72.263996,3.0,14289,False
4,cardboard102.jpg,cardboard,JPEG,RGB,512.0,384.0,48.388937,3.0,18015,False


In [80]:
df_images[df_images['corrompue'] == True]

,nom,classe,format,mode,largeur,hauteur,ecart_type_pixels,nb_canaux,taille_octets,corrompue
147,cardboard83.jpg,cardboard,NaN,NaN,NaN,NaN,NaN,NaN,20888,True
326,glass74.jpg,glass,NaN,NaN,NaN,NaN,NaN,NaN,7184,True
446,metal48.jpg,metal,NaN,NaN,NaN,NaN,NaN,NaN,7259,True
633,paper213.jpg,paper,NaN,NaN,NaN,NaN,NaN,NaN,7293,True
791,plastic13.jpg,plastic,NaN,NaN,NaN,NaN,NaN,NaN,5250,True
1004,trash3.jpg,trash,NaN,NaN,NaN,NaN,NaN,NaN,9192,True


## **Partie 2 – Détecter les images corrompues**

### **2.1 Ecriture d'une fonction qui détecte une image corrompue.**

In [81]:
def image_corrompu(file_path, classe):
    nom = os.path.basename(file_path)
    taille_octets = os.path.getsize(file_path)
    try:
        with Image.open(file_path) as img:
            img.load()  # exception levée si l'image est corrompue
            corrompu = False
    except Exception:
        corrompu = True

    return {
        "nom": nom,
        "taille": taille_octets,
        "classe": classe,
        "corrompu": corrompu,
    }

In [82]:
images_corrompu = []

for classe in sorted(os.listdir(RAW_DIR)):
    dossier_classe = os.path.join(RAW_DIR, classe)
    for nom_fichier in sorted(os.listdir(dossier_classe)):
        chemin_image = os.path.join(dossier_classe, nom_fichier)
        images_corrompu.append(image_corrompu(chemin_image, classe))

df_images_corrompu = pd.DataFrame(images_corrompu)
df_corrompues = df_images_corrompu[df_images_corrompu["corrompu"] == True]

print(f"Nombre total d'images : {len(df_images_corrompu)}")
print(f"Nombre d'images corrompues : {len(df_corrompues)}")
df_corrompues


Nombre total d'images : 1032
Nombre d'images corrompues : 6


,nom,taille,classe,corrompu
147,cardboard83.jpg,20888,cardboard,True
326,glass74.jpg,7184,glass,True
446,metal48.jpg,7259,metal,True
633,paper213.jpg,7293,paper,True
791,plastic13.jpg,5250,plastic,True
1004,trash3.jpg,9192,trash,True


## **Partie 3 – Détection des images vides**

### **3.1 Écrire et se servir d’une fonction qui détecte les images vides : image entièrement noire, image entièrement blanche ou image dont les pixels présentent très peu de variation.**

In [83]:
print(df_images['taille_octets'].min(), df_images['taille_octets'].max())

924 221825


In [84]:
def image_vide(file_path, classe, seuil_std=5):
    """Détecte une image vide : entièrement noire, entièrement blanche, ou à très faible variation."""
    nom = os.path.basename(file_path)

    with Image.open(file_path) as img:
        img.load()
        pixels = np.array(img)

    entierement_noire = pixels.max() == 0
    entierement_blanche = pixels.min() == 255
    faible_variation = pixels.std() < seuil_std

    return {
        "nom": nom,
        "classe": classe,
        "ecart_type_pixels": pixels.std(),
        "vide": entierement_noire or entierement_blanche or faible_variation,
    }

In [85]:
images_valides = df_images[df_images["corrompue"] == False]

resultats_vide = []
for _, ligne in images_valides.iterrows():
    chemin_image = os.path.join(RAW_DIR, ligne["classe"], ligne["nom"])
    resultats_vide.append(image_vide(chemin_image, ligne["classe"]))

df_vide = pd.DataFrame(resultats_vide)
df_images_vides = df_vide[df_vide["vide"] == True]

print(f"Nombre d'images vides : {len(df_images_vides)}")
df_images_vides

Nombre d'images vides : 2


,nom,classe,ecart_type_pixels,vide
166,image-blanche-512x384.jpg,cardboard,1.572536,True
355,image-blanche-512x384.jpg,metal,1.572536,True


## **Partie 4 – Détecter les différences de résolution**

### **4.1 Déterminer la résolution minimale, la résolution maximale, les résolutions les plus fréquentes et le nombre d'images par résolution**

In [88]:
images_valides = df_images[df_images["corrompue"] == False].copy()
images_valides['resolution'] = list(zip(images_valides['largeur'], images_valides['hauteur']))
images_valides['nb_pixels'] = images_valides['hauteur'] * images_valides['largeur']

images_valides.head()

,nom,classe,format,mode,largeur,hauteur,ecart_type_pixels,nb_canaux,taille_octets,corrompue,resolution,nb_pixels
0,cardboard1.jpg,cardboard,JPEG,RGB,512.0,384.0,40.588529,3.0,17333,False,"(512.0, 384.0)",196608.0
1,cardboard10.jpg,cardboard,JPEG,RGB,512.0,384.0,42.571288,3.0,21683,False,"(512.0, 384.0)",196608.0
2,cardboard100.jpg,cardboard,JPEG,RGB,512.0,384.0,46.108305,3.0,14884,False,"(512.0, 384.0)",196608.0
3,cardboard101.jpg,cardboard,JPEG,RGB,512.0,384.0,72.263996,3.0,14289,False,"(512.0, 384.0)",196608.0
4,cardboard102.jpg,cardboard,JPEG,RGB,512.0,384.0,48.388937,3.0,18015,False,"(512.0, 384.0)",196608.0


#### **la resolution minimale & maximale**

In [92]:
resolution_min = images_valides.loc[images_valides["nb_pixels"].idxmin(), "resolution"]
resolution_max = images_valides.loc[images_valides["nb_pixels"].idxmax(), "resolution"]

print("Résolution minimale :", resolution_min)
print("Résolution maximale :", resolution_max)


Résolution minimale : (32.0, 32.0)
Résolution maximale : (512.0, 384.0)


#### **les Résolutions les plus fréquentes**

In [96]:
images_valides['resolution'].mode()

0    (512.0, 384.0)
Name: resolution, dtype: object

#### **Nombre d'image par resolution**

In [95]:
images_valides.groupby('resolution')['resolution'].value_counts()

resolution
(32.0, 32.0)         5
(40.0, 40.0)         4
(48.0, 32.0)         4
(512.0, 384.0)    1013
Name: count, dtype: int64

### **4.1 On décide qu'une image doit avoir au minimum 64 × 64 pixels. Identifier toutes les images ne respectant pas cette contrainte.**

In [102]:
print(f"On a {len(images_valides[images_valides['resolution'] < (64, 64)])} images qui ne respecte pas cette condition")
images_valides[images_valides['resolution'] < (64, 64)]

On a 13 images qui ne respecte pas cette condition


,nom,classe,format,mode,largeur,hauteur,ecart_type_pixels,nb_canaux,taille_octets,corrompue,resolution,nb_pixels
20,cardboard117.jpg,cardboard,JPEG,RGB,48.0,32.0,63.599233,3.0,1708,False,"(48.0, 32.0)",1536.0
77,cardboard22.jpg,cardboard,JPEG,RGB,32.0,32.0,52.236024,3.0,1247,False,"(32.0, 32.0)",1024.0
133,cardboard70.jpg,cardboard,JPEG,RGB,40.0,40.0,63.887915,3.0,1773,False,"(40.0, 40.0)",1600.0
171,glass100.jpg,glass,JPEG,RGB,40.0,40.0,70.101908,3.0,1591,False,"(40.0, 40.0)",1600.0
229,glass15.jpg,glass,JPEG,RGB,48.0,32.0,17.324620,3.0,924,False,"(48.0, 32.0)",1536.0
268,glass21.jpg,glass,JPEG,RGB,32.0,32.0,51.574317,3.0,1349,False,"(32.0, 32.0)",1024.0
270,glass23.jpg,glass,JPEG,RGB,32.0,32.0,54.756708,3.0,1254,False,"(32.0, 32.0)",1024.0
383,metal121.jpg,metal,JPEG,RGB,48.0,32.0,35.567019,3.0,1169,False,"(48.0, 32.0)",1536.0
413,metal2.jpg,metal,JPEG,RGB,32.0,32.0,49.339948,3.0,1116,False,"(32.0, 32.0)",1024.0
421,metal26.jpg,metal,JPEG,RGB,40.0,40.0,62.174044,3.0,1667,False,"(40.0, 40.0)",1600.0
